### Import Libraries

In [1]:
import os
import yaml
from pathlib import Path
from typing import List
import warnings
warnings.filterwarnings('ignore')

from pydantic import BaseModel, Field
from crewai import Agent, Task, Crew, Process
from crewai import LLM
from crewai_tools import (FileReadTool, FileWriterTool, WebsiteSearchTool, SerperDevTool, ScrapeWebsiteTool)
from crewai.tools import tool, BaseTool

### Loading Tasks and Agents YAML Files

In [2]:
files = {
    'agents': 'config/agents.yaml',
    'tasks': 'config/tasks.yaml'
}

configs = {}
for config_type, file_path in files.items():
    with open(file_path, 'r') as file:
        configs[config_type] = yaml.safe_load(file)

agents_config = configs['agents']
tasks_config = configs['tasks']

### Initialize LLMs

In [3]:
WATSONX_URL = os.environ["WATSONX_URL"] = os.getenv("WATSONX_URL")  
WATSONX_APIKEY = os.environ["WATSONX_APIKEY"] = os.getenv("WATSONX_APIKEY") 
WATSONX_PROJECT_ID = os.environ["WATSONX_PROJECT_ID"] = os.getenv("WATSONX_PROJECT_ID")
WATSONX_MODEL_ID = os.environ["WATSONX_MODEL_ID"] = "watsonx/ibm/granite-3-2-8b-instruct-preview-rc"    # ibm/granite-3-8b-instruct , ibm/granite-3-2-8b-instruct-preview-rc, mistralai/mistral-large

llm = LLM(
    model = WATSONX_MODEL_ID,
    base_url = WATSONX_URL,
    project_id = WATSONX_PROJECT_ID,
	api_key = WATSONX_APIKEY,
    max_tokens = 10000,
    temperature = 0.1
)

### Define Tools

In [4]:
from langchain_community.tools.tavily_search import TavilySearchResults

class TavilyTool(BaseTool):
    name: str = "Tavily Search Tool"
    description: str = "Useful search engine built specifically for AI agents"
    search: TavilySearchResults = Field(default_factory=TavilySearchResults)
    
    def _run(self, query: str) -> str:
        try:
            return self.search.invoke(query)
        except Exception as e:
            return f"Error performing search: {str(e)}"

In [6]:
job_description_example = "job_description_example.md"
root_folder = Path("..").resolve()  
found_files = list(root_folder.rglob(job_description_example))
jd_sample_file = str(found_files[0])

In [8]:
web_search_tool = WebsiteSearchTool()
serper_dev_tool = SerperDevTool()
tavily_search_tool = TavilyTool()
file_read_tool = FileReadTool(file_path = jd_sample_file, description = 'A tool to read the job description example file.')
file_write_tool = FileWriterTool()

In [9]:
class ResearchRoleRequirements(BaseModel):
    """Research role requirements model"""
    skills: List[str] = Field(..., description="List of recommended skills for the ideal candidate aligned with the company's culture, ongoing projects, and the specific role's requirements.")
    experience: List[str] = Field(..., description="List of recommended experience for the ideal candidate aligned with the company's culture, ongoing projects, and the specific role's requirements.")
    qualities: List[str] = Field(..., description="List of recommended qualities for the ideal candidate aligned with the company's culture, ongoing projects, and the specific role's requirements.")

### Define Agents

In [10]:
research_agent = Agent(
    config = agents_config['research_agent'],
    tools = [web_search_tool, serper_dev_tool, tavily_search_tool],
    verbose = True,
    llm = llm,
    allow_delegation = False

)

writer_agent = Agent(
    config = agents_config['writer_agent'],
    tools = [web_search_tool, serper_dev_tool, tavily_search_tool, file_read_tool],
    verbose = True,
    llm = llm,
    allow_delegation = False
)
    
review_agent = Agent(
    config = agents_config['review_agent'],
    tools = [web_search_tool, serper_dev_tool, tavily_search_tool, file_read_tool, file_write_tool],
    verbose = True,
    llm = llm,
    allow_delegation = False
)

agents_list = [research_agent, writer_agent, review_agent]

### Define Tasks

In [11]:
research_company_culture_task = Task(
    config = tasks_config['research_company_culture_task'],
    agent = research_agent
)

research_role_requirements_task = Task(
    config = tasks_config['research_role_requirements_task'],
    agent = research_agent,
    output_json = ResearchRoleRequirements
)

draft_job_posting_task = Task(
    config = tasks_config['draft_job_posting_task'],
    agent = writer_agent
)

review_and_edit_job_posting_task = Task(
    config = tasks_config['review_and_edit_job_posting_task'],
    agent = review_agent,
    output_file = 'new_job_post.md'  
)


industry_analysis_task = Task(
    config = tasks_config['industry_analysis_task'],
    agent = research_agent
)

tasks_list = [research_company_culture_task, research_role_requirements_task, industry_analysis_task, draft_job_posting_task, review_and_edit_job_posting_task]

### Run Crew

In [12]:
jd_creation_crew = Crew(
    agents = agents_list,  
    tasks= tasks_list,  
    process = Process.sequential,
    verbose = True
)

In [13]:
inputs = {
    'company_domain': 'www.ibm.com',
    'company_description': "IBM is a global leader in hybrid cloud, AI, and consulting. With WatsonX, IBM is revolutionizing AI-powered solutions, helping enterprises scale AI with trusted data and governance. The Client Engineering team collaborates with clients to co-create AI-driven innovations.",
    'hiring_needs': 'AI Engineer for WatsonX Client Engineering team in Bengaluru, India, responsible for building and deploying AI solutions for enterprise clients.',
    'specific_benefits': 'Competitive salary, flexible work environment, professional development programs, healthcare, and global networking opportunities.',
}

result = jd_creation_crew.kickoff(inputs=inputs)

# Agent: Research Analyst
## Task: Analyze the provided company website and the hiring manager's company's domain www.ibm.com, description IBM is a global leader in hybrid cloud, AI, and consulting. With WatsonX, IBM is revolutionizing AI-powered solutions, helping enterprises scale AI with trusted data and governance. The Client Engineering team collaborates with clients to co-create AI-driven innovations.. Focus on understanding the company's culture, values, and mission. Identify unique selling points and specific projects or achievements highlighted on the site. Compile a report summarizing these insights, specifically how they can be leveraged in a job posting to attract the right candidates.



Inserting batches in chromadb: 100%|██████████| 1/1 [00:01<00:00,  1.35s/it]
2025-02-13 14:52:37,245 - 8378665024 - local_persistent_hnsw.py-local_persistent_hnsw:423 - WARNING: Number of requested results 3 is greater than number of elements in index 2, updating n_results = 2




# Agent: Research Analyst
## Thought: Thought: I need to analyze the IBM website to understand its culture, values, mission, and unique selling points. I will use the 'Search in a specific website' tool to extract relevant information.
## Using tool: Search in a specific website
## Tool Input: 
"{\"search_query\": \"IBM culture\", \"website\": \"https://www.ibm.com\"}"
## Tool Output: 
Relevant Content:
IBM - India Maximize Productivity with AI Agents and Responsible AI Deploy Agentic AI to automate complex tasks, optimize workflows, and drive enterprise-wide efficiency Read the full article Explore watsonx Orchestrate Latest news February 27, 2025: Integration Evolution: From Hybrid Cloud to Intelligent Automation Indian Companies are Investing in AI for the Long Term, Intend to Use Open-Source Tools to Drive ROI and Innovation: IBM Study AI Outlook for 2025: New Study Finds Decisive Shift as Indian Enterprises Move Beyond Early Experimentation to Place ROI at the Heart of AI IBM’s 

2025-02-13 14:53:07,630 - 8378665024 - local_persistent_hnsw.py-local_persistent_hnsw:423 - WARNING: Number of requested results 3 is greater than number of elements in index 2, updating n_results = 2




# Agent: Research Analyst
## Thought: Thought: I need to analyze the company culture, values, and mission to identify the key skills, experiences, and qualities required for the AI Engineer role in the WatsonX Client Engineering team in Bengaluru, India. I will use the provided context and the available tools to gather information.
## Using tool: Search in a specific website
## Tool Input: 
"{\"search_query\": \"AI Engineer job requirements for IBM WatsonX Client Engineering team\", \"website\": \"https://www.ibm.com\"}"
## Tool Output: 
Relevant Content:
leaders Deploy an AI agent for HR with watsonx Orchestrate Protect your data with IBM Guardium Data Security Center How clients are applying AI See how IBM clients are problem solving their way to real business value with AI. Discover how companies are using watsonx Technology & Consulting From next-generation AI to cutting edge hybrid cloud solutions to the deep expertise of IBM Consulting, IBM has what it takes to help you reinven

Inserting batches in chromadb: 100%|██████████| 1/1 [00:00<00:00,  1.43it/s]




# Agent: Research Analyst
## Thought: Thought: I need to gather information about the industry trends, challenges, and opportunities related to AI and machine learning, focusing on IBM's domain. I will use the Search in a specific website tool to analyze IBM's website and the Tavily Search Tool for broader industry insights.
## Using tool: Search in a specific website
## Tool Input: 
"{\"search_query\": \"AI trends in IBM\", \"website\": \"www.ibm.com\"}"
## Tool Output: 
Relevant Content:
IBM - India Maximize Productivity with AI Agents and Responsible AI Deploy Agentic AI to automate complex tasks, optimize workflows, and drive enterprise-wide efficiency Read the full article Explore watsonx Orchestrate Latest news February 27, 2025: Integration Evolution: From Hybrid Cloud to Intelligent Automation Indian Companies are Investing in AI for the Long Term, Intend to Use Open-Source Tools to Drive ROI and Innovation: IBM Study AI Outlook for 2025: New Study Finds Decisive Shift as Ind

In [14]:
print(result.raw)

# Job Posting: AI Engineer - WatsonX Client Engineering Team

## About IBM

IBM is a global leader in hybrid cloud, AI, and consulting, revolutionizing the way businesses operate through innovative technology solutions. With WatsonX, IBM is at the forefront of AI-powered solutions, empowering enterprises to scale AI with trusted data and governance. Our Client Engineering team collaborates closely with clients to co-create AI-driven innovations that drive business value and transform industries.

## Role Overview

We are seeking a talented and passionate AI Engineer to join our WatsonX Client Engineering team in Bengaluru, India. As an AI Engineer, you will play a pivotal role in building and deploying cutting-edge AI solutions for our enterprise clients, leveraging IBM's expertise in hybrid cloud environments and responsible AI practices.

## Responsibilities

1. Design, develop, and deploy AI models and solutions for enterprise clients, utilizing IBM WatsonX and agentic AI.
2. Collab

In [ ]:
from IPython.display import display, Markdown
display(Markdown(result.raw))

In [ ]:
import pandas as pd

costs = 0.150 * (jd_creation_crew.usage_metrics.prompt_tokens + jd_creation_crew.usage_metrics.completion_tokens) / 1_000_000
print(f"Total costs: ${costs:.4f}")

# Convert UsageMetrics instance to a DataFrame
df_usage_metrics = pd.DataFrame([jd_creation_crew.usage_metrics.dict()])
df_usage_metrics